# Warsztat 4 — Rozmowa z kupcem sterowanym przez LLM

**Cel:** zbudować tekstowy dialog z kupcem w grze. Gracz pyta o towar i prosi o zakup mikstury.

**Podział odpowiedzialności (kluczowa idea warsztatu):**

| Model językowy (LLM) | Program (logika gry) |
|---|---|
| prowadzi swobodną rozmowę | przechowuje saldo, cenę i zapas |
| rozpoznaje intencję gracza | waliduje JSON i listę dozwolonych akcji |
| **proponuje** odpowiedź i akcję (`talk` / `buy_potion`) | **rozstrzyga**, czy zakup jest możliwy, i zmienia stan |
| — | tworzy komunikat potwierdzający transakcję |

Stan początkowy: gracz ma **15 monet**, kupiec **2 mikstury**, cena **10 monet**.
Niepoprawny JSON, nieznana akcja lub timeout **nie zmieniają stanu gry**.

### Jak pracować z tym notebookiem
- Miejsca do uzupełnienia są oznaczone **`TO BE DONE n/6`** (każde to 1–6 linii kodu) — szukaj ich też po `>>> TO BE DONE`.
- Pod każdym zadaniem jest komórka **„Sprawdzenie”** z testami — działa bez modelu, uruchom ją od razu po uzupełnieniu.
- Komórki bez oznaczenia są dostarczone — nie trzeba ich zmieniać
- Kolejność: 0–10 min podział odpowiedzialności i nazwa modelu → 10–30 TODO 1–2 → 30–55 TODO 3–5 → 55–75 TODO 6 i sześć prób → 75–90 wnioski.

## 0. Ustawienia i połączenie z modelem

W pierwszej komórce wybierz dostawcę modelu i skonfiguruj połączenie.

| Ustawienie | Ollama | API zgodne z OpenAI (przykład Groq) |
|---|---|---|
| `MODEL` | `llama3.2:3b` | `openai/gpt-oss-20b` |
| `MODEL_URL` | `http://localhost:11434/api/chat` | `https://api.groq.com/openai/v1` |
| `USE_OLLAMA` | `True` | `False` |
| `API_KEY` | Nie jest wymagany | Klucz API ze zmiennej środowiskowej `API_KEY` lub pliku `.env` |

**Wybór dostawcy:**
- `USE_OLLAMA = True` – korzystanie z lokalnego modelu przez Ollamę.
- `USE_OLLAMA = False` – korzystanie z zewnętrznego API zgodnego z OpenAI, np. Groq, OpenRouter lub innego obsługiwanego endpointu.

**Konfiguracja API:**

Dla usług zewnętrznych ustaw odpowiednią nazwę modelu oraz adres bazowy API. Klucz API można przekazać przez zmienną środowiskową `API_KEY` lub umieścić w pliku `.env` znajdującym się w katalogu roboczym notatnika:

`API_KEY=twój_klucz_api`

Nie umieszczaj prawdziwych kluczy API bezpośrednio w notatniku ani w repozytorium.

Przykładowo, klucz do Groq można uzyskać na stronie: [Groq API Keys](https://console.groq.com/keys).
Należy zalogować się (można użyć prywatnego lub uczelnianego konta Google), a następnie wygenerować nowy klucz API. Po zakończeniu zajęć można go usunąć w ustawieniach konta Groq.

Uwaga – limity Groq: Darmowy dostęp do API ma ograniczenia liczby zapytań i wykorzystania tokenów. Po przekroczeniu limitu może pojawić się błąd 429 Too Many Requests lub rate_limit_exceeded. W takiej sytuacji zazwyczaj należy odczekać około minuty aby ponownie wysyłać request.

**Konfiguracja Ollamy:**

Uruchom Ollamę (aplikację lub `ollama serve`) i pobierz wybrany model, np.:

`ollama pull llama3.2:3b`

Następnie pozostaw `USE_OLLAMA = True`.

**Pozostałe ustawienia:**

- `TEMPERATURE` – określa losowość generowanych odpowiedzi.
- `SEED` – ziarno losowości, jeśli jest obsługiwane przez wybranego dostawcę.
- `TIMEOUT_S` – maksymalny czas oczekiwania na odpowiedź modelu w sekundach.
- `USE_MOCK` – włącza atrapę modelu. Przydatne do sprawdzenia działania logiki gry bez rzeczywistego połączenia z LLM. Wyniki uzyskane w tym trybie nie pochodzą z prawdziwego modelu.

W sprawozdaniu zapisz nazwę dostawcy, modelu oraz wykorzystane ustawienia generowania.

In [ ]:
import json, os, urllib.request, urllib.error
from pathlib import Path

# ====== USTAWIENIA (zapisz je w sprawozdaniu) ======
MODEL       = "llama3.2:3b"                         # nazwa modelu (zapisz w sprawozdaniu!)
MODEL_URL  = "http://localhost:11434/api/chat"
USE_OLLAMA = True                                          # True = używamy Ollama (lokalny LLM), False = używamy API (Groq, OpenRouter, Gemini, LM Studio, Custom)
# przykładowe ustawienia dla groq api
# MODEL       = "openai/gpt-oss-20b"                         
# MODEL_URL   = "https://api.groq.com/openai/v1"  
  
API_KEY     = os.getenv("API_KEY")                  # używany tylko przez API zgodne z OpenAI
TEMPERATURE = 0.3                                   # ustawienia generowania
SEED        = 42
TIMEOUT_S   = 60                                    # po tylu sekundach uznajemy, że model nie odpowiedział
USE_MOCK    = False                                 # True = atrapa modelu (gdy Ollama nie działa i nie ma API_KEY); wyniki NIE są wtedy z prawdziwego LLM

MODEL_URL = MODEL_URL.rstrip("/")

env_file = Path(".env")
if not USE_OLLAMA and not API_KEY and env_file.is_file():
    for line in env_file.read_text(encoding="utf-8-sig").splitlines():
        name, separator, value = line.partition("=")
        if separator and name.strip() == "API_KEY":
            API_KEY = value.strip().strip("'\"")
            break

ALLOWED_ACTIONS = ["talk", "buy_potion"]            # jedyne dozwolone propozycje modelu
PRICE = 10

In [ ]:
import re

def mock_llm(user_text : str) -> str:
    """Prosta atrapa LLM do testów offline (zastępuje model, gdy USE_MOCK=True)."""
    t = user_text.lower()
    if re.search(r"poprosz|kupi|biorę|biore|wezmę", t) and "miecz" not in t:
        out = {"reply": "Dobry wybór, wędrowcze. Oto mikstura!", "action": "buy_potion"}
    elif "miecz" in t:
        out = {"reply": "Mieczy nie skupuję, wędrowcze.", "action": "talk"}
    else:
        out = {"reply": "Mam mikstury leczące po 10 monet za sztukę.", "action": "talk"}
    return json.dumps(out, ensure_ascii=False)


def call_llm(system_prompt, user_text):
    """Zwraca surowy tekst odpowiedzi modelu. Przy timeoucie / błędzie sieci rzuca wyjątek."""
    if USE_MOCK:
        return mock_llm(user_text)
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_text},
    ]
    if not USE_OLLAMA:
        if not API_KEY:
            raise ValueError("Ustaw API_KEY w zmiennej środowiskowej lub w pliku .env.")
        from openai import OpenAI

        with OpenAI(base_url=MODEL_URL, api_key=API_KEY, timeout=TIMEOUT_S, max_retries=0) as client:
            response = client.chat.completions.create(
                model=MODEL, messages=messages, temperature=TEMPERATURE,
                    response_format={
                    "type": "json_schema",
                    "json_schema": {
                        "name": "merchant_response",
                        "strict": True,
                        "schema": RESPONSE_SCHEMA
                    }
                }   
            )
        return response.choices[0].message.content

    payload = {
        "model": MODEL,
        "stream": False,
        "format": RESPONSE_SCHEMA or "json",   # odpowiedź zgodna ze schematem (structured output w Ollama)
        "messages": messages,
        "options": {"temperature": TEMPERATURE, "seed": SEED},
    }
    req = urllib.request.Request(MODEL_URL, data=json.dumps(payload).encode("utf-8"),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=TIMEOUT_S) as r:
        return json.loads(r.read())["message"]["content"]

In [ ]:
# --- TEST POŁĄCZENIA (dostarczony): sprawdza połączenie i dla ollamy ładuje model do pamięci (pierwsze wywołanie bywa wolne) ---
if USE_MOCK:
    print("Tryb atrapy (USE_MOCK=True) — prawdziwy model nie jest używany.")
elif not USE_OLLAMA:
    try:
        call_llm('Odpowiedz obiektem JSON z polami "reply" i "action".', "Hej!")
        print(f"[OK] Model {MODEL} odpowiada.")
    except Exception as e:
        error = "Ustaw API_KEY w zmiennej środowiskowej lub w pliku .env." if not API_KEY else type(e).__name__
        print(f"[BŁĄD] {error}")
else:
    try:
        tags = json.loads(urllib.request.urlopen(MODEL_URL.replace("/api/chat", "/api/tags"), timeout=5).read())
        names = [m["name"] for m in tags["models"]]
        print("Ollama działa. Pobrane modele:", names)
        if MODEL not in names:
            print(f"[UWAGA] Brak modelu {MODEL!r} — w terminalu: ollama pull {MODEL}")
        else:
            body = json.dumps({"model": MODEL, "prompt": "hej", "stream": False, "options": {"num_predict": 1}}).encode()
            urllib.request.urlopen(urllib.request.Request(MODEL_URL.replace("/api/chat", "/api/generate"), data=body,
                                   headers={"Content-Type": "application/json"}), timeout=300).read()
            print(f"[OK] Model {MODEL} załadowany i odpowiada.")
    except urllib.error.URLError:
        print("[BŁĄD] Ollama nie odpowiada. Uruchom aplikację Ollama albo w terminalu: ollama serve")

In [ ]:
# --- STAN GRY, POSTACIE I SCENARIUSZE TESTÓW ---
from typing import Any

from typing import TypedDict, Any

class State(TypedDict):
    coins: Any
    stock: Any
    price: Any

class Scenerio(TypedDict):
    id : int
    name : str
    start : State
    user: str
    expected_action: str
    expected_state : State

def make_state(coins : int, stock : int) -> State:
    return {"coins": coins, "stock": stock, "price": PRICE}

PERSONAS : dict[str,str] = {
    "Borys":  "Jesteś Borys, rzeczowy i trochę zrzędliwy kupiec. Mówisz krótko, konkretnie, bez ozdobników.",
    "Lirael": "Jesteś Lirael, poetycka kupcowa-wróżka. Mówisz kwieciście, pełnymi metafor zdaniami, ale zawsze dochodzisz do sedna.",
}

SCENARIOS : Scenerio= [
    {"id": 1, "name": "Pytanie o ofertę",           "start": make_state(15, 2), "user": "Co masz dzisiaj na sprzedaż i ile to kosztuje?",
     "expected_action": "talk",       "expected_state": make_state(15, 2)},
    {"id": 2, "name": "Poprawny zakup",             "start": make_state(15, 2), "user": "Poproszę jedną miksturę.",
     "expected_action": "buy_potion", "expected_state": make_state(5, 1)},
    {"id": 3, "name": "Zakup bez środków",          "start": None,              "user": "Poproszę jeszcze jedną miksturę.",   # None = kontynuacja testu 2
     "expected_action": "buy_potion", "expected_state": make_state(5, 1)},
    {"id": 4, "name": "Pusty magazyn",              "start": make_state(50, 0), "user": "Chcę kupić miksturę.",
     "expected_action": "buy_potion", "expected_state": make_state(50, 0)},
    {"id": 5, "name": "Niejasna wypowiedź",         "start": make_state(15, 2), "user": "Hmm... sam nie wiem, może coś na te moje bóle w kolanach?",
     "expected_action": "talk",       "expected_state": make_state(15, 2)},
    {"id": 6, "name": "Nieobsługiwana czynność",    "start": make_state(15, 2), "user": "Sprzedam ci mój miecz za 20 monet.",
     "expected_action": "talk",       "expected_state": make_state(15, 2)},
]

## TO BE DONE 1/6 — instrukcja dla modelu (prompt systemowy)
Instrukcja ma opisywać rolę kupca, **aktualny stan** i **dozwolone akcje**. Zwróć uwagę: stan trafia do promptu tylko informacyjnie — model go nie zmienia.

In [ ]:
def build_system_prompt(state : State, persona : str) -> str:
    lines = [PERSONAS[persona],
             "Jesteś kupcem w grze fabularnej i rozmawiasz z graczem po polsku."]

    # >>> TO BE DONE 1/6 — dopisz do `lines` kolejne elementy instrukcji:
    #   a) aktualny stan gry z `state` (monety gracza, zapas mikstur, cena) — użyj f-stringa,
    #   b) dozwolone akcje "talk" i "buy_potion" z opisem, kiedy użyć której (buy_potion tylko przy WYRAŹNEJ prośbie o zakup),
    #   c) zasadę: model tylko PROPONUJE, o transakcji decyduje program,
    #   d) format odpowiedzi: wyłącznie obiekt JSON, np. {"action": "talk" lub "buy_potion", "reply": "<wypowiedź kupca>"}.
    #   Wskazówka: małe modele lubią dopytywać zamiast proponować zakup — pomaga 1 linia z przykładami.
    # lines += [ ..., ..., ... ]
    # <<<

    return "\n".join(lines)

In [ ]:
# Sprawdzenie TODO 1
p = build_system_prompt(make_state(15, 2), "Borys")
assert "15" in p and "2" in p and "10" in p, "W instrukcji brakuje stanu gry (monety / zapas / cena)"
assert "talk" in p and "buy_potion" in p, "W instrukcji brakuje dozwolonych akcji"
assert "reply" in p and "action" in p and "JSON" in p, "W instrukcji brakuje opisu formatu JSON"
print("[OK] TODO 1 OK — instrukcja zawiera stan, akcje i format.\n")
print(p)

## TO BE DONE 2/6 — format odpowiedzi (JSON Schema)
Zdefiniuj schemat, który zostanie wykorzystany do wymuszenia struktury odpowiedzi (`format` w `call_llm`). Pole `action` ogranicz do dozwolonych wartości.

In [ ]:
RESPONSE_SCHEMA = None

# >>> TO BE DONE 2/6 (ok. 5-6 linii) — zdefiniuj JSON Schema odpowiedzi modelu jako słownik Pythona:
#   - "type": "object"
#   - "properties": "action" (typ string, ograniczony przez "enum" do ALLOWED_ACTIONS) oraz "reply" (typ string)
#     Uwaga: model generuje pola w kolejności ze schematu — daj "action" PRZED "reply" (najpierw decyzja, potem wypowiedź).
#   - "required": oba pola
#   - "additionalProperties": False
# RESPONSE_SCHEMA = { ... }
# <<<

In [ ]:
# Sprawdzenie TODO 2
s = RESPONSE_SCHEMA
assert isinstance(s, dict) and s.get("type") == "object", "Schemat musi być słownikiem z type=object"
assert set(s["properties"]) == {"reply", "action"}, "properties musi zawierać dokładnie reply i action"
assert s["properties"]["action"].get("enum") == ALLOWED_ACTIONS, "action musi mieć enum = ALLOWED_ACTIONS"
assert set(s["required"]) == {"reply", "action"}, "required: oba pola"
print("[OK] TODO 2 OK — schemat odpowiedzi gotowy.")

## TO BE DONE 3/6 — walidacja formatu odpowiedzi
Nawet ze schematem warto sprawdzić odpowiedź po stronie programu (inne modele/usługi mogą schematu nie wspierać). Niepoprawny JSON **nie zmienia stanu**.

In [ ]:
def parse_model_output(raw : str) -> tuple[dict | None, str]:
    """Zwraca (dane, None) gdy format poprawny albo (None, 'opis błędu')."""
    try:
        data = json.loads(raw)
    except (json.JSONDecodeError, TypeError):
        return None, "niepoprawny JSON"
    
    # >>> TO BE DONE 3/6 — walidacja pól; przy każdym problemie `return None, "krótki opis błędu"`:
    #   a) `data` musi być słownikiem (dict),
    #   b) musi zawierać klucze "reply" i "action",
    #   c) "reply" to niepusty string, "action" to string.
    # <<<

    return data, None

In [ ]:
# Sprawdzenie TODO 3
good = '{"reply": "Witaj!", "action": "talk"}'
assert parse_model_output(good)[0] == {"reply": "Witaj!", "action": "talk"}, "poprawny JSON powinien przejść"
for bad in ['to nie jest json', '[1, 2]', '{"reply": "x"}', '{"action": "talk"}', '{"reply": 5, "action": "talk"}',
            '{"reply": "  ", "action": "talk"}', '{"reply": "x", "action": 1}', None]:
    d, err = parse_model_output(bad)
    assert d is None and err, f"to powinno zostać odrzucone: {bad!r}"
print("[OK] TODO 3 OK — parser odrzuca niepoprawny format.")

## TO BE DONE 4/6 — lista dozwolonych akcji
Poprawny JSON nadal może zawierać nieznaną akcję (np. `steal`). Taką propozycję trzeba odrzucić.

In [ ]:
def validate_proposal(data : dict[str, Any]) -> tuple[bool, str | None]:
    """Sprawdza, czy proponowana akcja jest na liście dozwolonych. Zwraca (True, None) lub (False, 'powód')."""

    # >>> TO BE DONE 4/6 — jeśli data["action"] NIE jest w ALLOWED_ACTIONS,
    #     zwróć (False, "nieznana akcja: ...") — nie poprawiaj akcji samodzielnie, tylko odrzuć.
    # <<<

    return True, None

In [ ]:
# Sprawdzenie TODO 4
assert validate_proposal({"reply": "x", "action": "talk"}) == (True, None)
assert validate_proposal({"reply": "x", "action": "buy_potion"}) == (True, None)
for a in ["steal", "sell_sword", "BUY_POTION", ""]:
    ok, err = validate_proposal({"reply": "x", "action": a})
    assert ok is False and err, f"akcja {a!r} powinna zostać odrzucona"
print("[OK] TODO 4 OK — nieznane akcje są odrzucane.")

## TO BE DONE 5/6 — wykonanie zakupu przez program
Serce warsztatu: program **sam** sprawdza monety i zapas i dopiero wtedy zmienia stan. Komunikat transakcji powstaje tutaj, a nie w wypowiedzi modelu.

In [ ]:
from typing import Literal


def execute_action(state : State, action : Literal["talk", "buy_potion"]):
    """Wykonuje akcję na stanie gry. Zwraca (nowy_stan, czy_zaakceptowano, komunikat_programu). Nie modyfikuje `state`."""
    if action == "talk":
        return dict(state), True, ""
    if action == "buy_potion":
        # >>> TO BE DONE 5/6 — to JEDYNE miejsce, które zmienia stan gry:
        #   a) brak mikstur (state["stock"] < 1)            -> return dict(state), False, "<komunikat odrzucenia>"
        #   b) za mało monet (state["coins"] < state["price"]) -> return dict(state), False, "<komunikat odrzucenia>"
        #   c) w przeciwnym razie: zbuduj NOWY stan (monety - cena, zapas - 1; użyj {**state, ...})
        #      i zwróć (nowy_stan, True, "<komunikat sukcesu z liczbami z nowego stanu>")
        # <<<
        pass
    return dict(state), False, f"Nieobsługiwana akcja: {action}"

In [ ]:
# Sprawdzenie TODO 5
s0 = make_state(15, 2)
s1, ok, msg = execute_action(s0, "buy_potion")
assert ok and (s1["coins"], s1["stock"]) == (5, 1) and msg, "poprawny zakup: 15->5 monet, 2->1 mikstura, niepusty komunikat"
assert (s0["coins"], s0["stock"]) == (15, 2), "funkcja nie może modyfikować stanu wejściowego"
s2, ok, msg = execute_action(s1, "buy_potion")
assert not ok and (s2["coins"], s2["stock"]) == (5, 1) and msg, "za mało monet: stan bez zmian, zakup odrzucony"
s3, ok, msg = execute_action(make_state(50, 0), "buy_potion")
assert not ok and (s3["coins"], s3["stock"]) == (50, 0) and msg, "pusty magazyn: stan bez zmian, zakup odrzucony"
assert execute_action(s0, "talk")[0] == s0
print("[OK] TODO 5 OK — program sam pilnuje monet i zapasu.")

## TO BE DONE 6/6 — ocena trzech poziomów poprawności
Oddzielamy: **format** (czy JSON poprawny), **intencja** (czy propozycja modelu była właściwa) i **stan** (czy gra jest w poprawnym stanie).

In [ ]:
def evaluate(rec, scenario : Scenerio):
    """Trzy OSOBNE oceny: format, intencja, stan gry. Zwraca słownik z kluczami format_ok, intent_ok, state_ok."""
    # >>> TO BE DONE 6/6 — porównaj wynik próby `rec` z oczekiwaniami `scenario`:
    #   - format_ok: rec["format_ok"]
    #   - intent_ok: czy rec["proposed_action"] == scenario["expected_action"]
    #   - state_ok : czy monety i zapas w rec["state_after"] == te w scenario["expected_state"]
    #                (porównaj tylko klucze "coins" i "stock")
    # return {"format_ok": ..., "intent_ok": ..., "state_ok": ...}
    # <<<
    raise NotImplementedError("TODO 6")

In [ ]:
# Sprawdzenie TODO 6
sc = SCENARIOS[1]   # poprawny zakup
good = {"format_ok": True, "proposed_action": "buy_potion", "state_after": make_state(5, 1)}
bad  = {"format_ok": True, "proposed_action": "talk",       "state_after": make_state(15, 2)}
assert evaluate(good, sc) == {"format_ok": True, "intent_ok": True, "state_ok": True}
assert evaluate(bad, sc)  == {"format_ok": True, "intent_ok": False, "state_ok": False}
print("[OK] TODO 6 OK — trzy poziomy poprawności oceniane osobno.")

## Pętla tury (dostarczona)
Połączenie wszystkich elementów: model → walidacja formatu → walidacja akcji → decyzja programu.

In [ ]:
# --- PĘTLA JEDNEJ TURY (dostarczona): łączy model z logiką gry ---
def run_turn(state : State, user_text : str, persona="Borys"):
    rec = {"user": user_text, "persona": persona, "state_before": dict(state), "raw": None, "reply": None,
           "proposed_action": None, "format_ok": False, "decision": "", "message": "", "state_after": dict(state)}

    # 1) wywołanie modelu — błąd / timeout NIE zmienia stanu
    try:
        rec["raw"] = call_llm(build_system_prompt(state, persona), user_text)
    except Exception as e:
        rec["decision"] = f"ODRZUCONO (błąd/timeout: {type(e).__name__})"
        rec["message"] = "(kupiec milczy)"
        return rec

    # 2) poprawność formatu
    data, err = parse_model_output(rec["raw"])
    if data is None:
        rec["decision"] = f"ODRZUCONO (zły format: {err})"
        rec["message"] = "(kupiec mamrocze coś niezrozumiale)"
        return rec
    rec["format_ok"], rec["reply"], rec["proposed_action"] = True, data["reply"], data["action"]

    # 3) lista dozwolonych akcji
    ok, err = validate_proposal(data)
    if not ok:
        rec["decision"] = f"ODRZUCONO ({err})"
        rec["message"] = "(kupiec nie rozumie, o co chodzi)"
        return rec

    # 4) wykonanie akcji przez PROGRAM; komunikat transakcji pochodzi z wyniku programu, nie z modelu
    new_state, accepted, info = execute_action(state, data["action"])
    rec["state_after"] = new_state
    if data["action"] == "talk":
        rec["decision"], rec["message"] = "OK (rozmowa)", data["reply"]
    else:
        rec["decision"] = "ZAKUP ZAAKCEPTOWANY" if accepted else "ZAKUP ODRZUCONY"
        rec["message"] = info
    return rec


def show_turn(rec):
    print(f"Gracz       : {rec['user']}")
    print(f"Model (raw) : {rec['raw']}")
    print(f"Propozycja  : {rec['proposed_action']}   | decyzja programu: {rec['decision']}")
    print(f"Kupiec      : {rec['message']}")
    print(f"Stan        : {rec['state_before']['coins']} mon., {rec['state_before']['stock']} mikst.  ->  "
          f"{rec['state_after']['coins']} mon., {rec['state_after']['stock']} mikst.\n")

In [ ]:
# --- URUCHAMIANIE SZEŚCIU PRÓB I ZAPIS WYNIKÓW (dostarczone) ---
def run_scenarios(persona="Borys", scenarios=SCENARIOS, verbose=True):
    results, prev_state = [], None
    for sc in scenarios:
        start = sc["start"] if sc["start"] is not None else prev_state     # test 3 kontynuuje test 2
        rec = run_turn(start, sc["user"], persona)
        rec.update(evaluate(rec, sc))
        rec["id"], rec["name"] = sc["id"], sc["name"]
        prev_state = rec["state_after"]
        results.append(rec)
        if verbose:
            print(f"=== Próba {sc['id']}: {sc['name']} ({persona}) ===")
            show_turn(rec)
    print_table(results)
    fname = f"wyniki_{persona}.json"
    with open(fname, "w", encoding="utf-8") as f:
        json.dump({"model": MODEL, "temperature": TEMPERATURE, "seed": SEED, "mock": USE_MOCK, "persona": persona,
                   "results": results}, f, ensure_ascii=False, indent=2)
    print(f"Zapisano: {fname}")
    return results


def print_table(results):
    mark = lambda b: "OK " if b else "BŁĄD"
    print(f"{'#':<2} {'Próba':<26} {'Propozycja':<11} {'Decyzja programu':<32} {'Monety':<7} {'Zapas':<6} {'Format':<7} {'Intencja':<9} {'Stan'}")
    for r in results:
        print(f"{r['id']:<2} {r['name']:<26} {str(r['proposed_action']):<11} {r['decision'][:31]:<32} "
              f"{r['state_after']['coins']:<7} {r['state_after']['stock']:<6} "
              f"{mark(r['format_ok']):<7} {mark(r['intent_ok']):<9} {mark(r['state_ok'])}")
    print("\nFormat = poprawny JSON z polami | Intencja = model zaproponował oczekiwaną akcję | Stan = końcowy stan gry zgodny z oczekiwanym")

## Sześć prób
Pytanie o ofertę · poprawny zakup · ponowny zakup bez środków (kontynuacja poprzedniego) · pusty magazyn · niejasna wypowiedź · nieobsługiwana czynność.

In [ ]:
results = run_scenarios("Borys")

## Zakres dodatkowy — druga postać

Ta sama logika transakcji, inny styl wypowiedzi (Borys kontra Lirael). Sprawdź, czy zmiana stylu wpływa na **poprawność akcji**
(a nie tylko na brzmienie odpowiedzi).

In [ ]:
res_b = run_scenarios("Borys", verbose=False)
res_l = run_scenarios("Lirael", verbose=False)

print("\nPorównanie propozycji akcji i poprawności:")
print(f"{'#':<2} {'Borys':<12} {'Lirael':<12} {'Borys (F/I/S)':<15} {'Lirael (F/I/S)'}")
flags = lambda r: "/".join("T" if r[k] else "N" for k in ("format_ok", "intent_ok", "state_ok"))
for b, l in zip(res_b, res_l):
    print(f"{b['id']:<2} {str(b['proposed_action']):<12} {str(l['proposed_action']):<12} {flags(b):<15} {flags(l)}")
print("\nPrzykładowe wypowiedzi z próby 2 (zakup):")
print("Borys :", res_b[1]["raw"])
print("Lirael:", res_l[1]["raw"])

## Demo na forum (5–6 min)
Pokaż zakup dozwolony i odrzucony: propozycję modelu (`[propozycja modelu: ...]`) oraz faktyczny stan gry, i wyjaśnij rolę walidacji.

In [ ]:
# --- Interaktywna rozmowa (do dema na forum). Wpisz "koniec", aby wyjść. ---
def chat(persona="Borys", state=None):
    state = state or make_state(15, 2)
    print(f"Rozmawiasz z kupcem: {persona}. Masz {state['coins']} monet. (wpisz 'koniec' aby wyjść)\n")
    while True:
        text = input("Ty: ")
        if text.strip().lower() == "koniec":
            break
        rec = run_turn(state, text, persona)
        state = rec["state_after"]
        print(f"[propozycja modelu: {rec['proposed_action']} | {rec['decision']}]")
        print(f"{persona}: {rec['message']}")
        print(f"[stan gry: {state['coins']} monet, {state['stock']} mikstur]\n")

# chat("Borys")      # <- odkomentuj, żeby porozmawiać

## Sprawozdanie i wnioski

**Model:** *(wpisz nazwę modelu)*  |  **Ustawienia:** *(temperature, seed, timeout)*
Uwaga: przy ponownym uruchomieniu tekst odpowiedzi może się różnić.

**Tabela sześciu testów:** wklej tabelę wydrukowaną powyżej (lub dołącz plik `wyniki_Borys.json`).

**Wnioski** — w kilku zdaniach, **oddzielnie** dla trzech poziomów:
1. Poprawność formatu: *(...)*
2. Rozpoznanie intencji (czy propozycja modelu była właściwa — nawet zgodna ze schematem odpowiedź może być błędna): *(...)*
3. Poprawność zmiany stanu gry (co zapewnił program, a czego nie mógłby zapewnić sam model): *(...)*
4. Czy zmiana postaci (Borys → Lirael) wpłynęła na poprawność akcji? *(...)*